# Fase 2 — Limpieza y Transformación del Dataset
**Dataset:** Olist Brazilian E-Commerce  
**Input:** CSVs originales en `data/raw/`  
**Output:** CSVs limpios en `data/clean/`  

Cada decisión de limpieza está justificada en base a los hallazgos de `01_exploration.ipynb`.

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import os
import unicodedata
import re
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Crear carpeta de salida si no existe
os.makedirs('../data/clean', exist_ok=True)

def log(msg):
    print(f'  ✔  {msg}')

def section(title):
    print(f'\n{'='*60}\n  {title}\n{'='*60}')

def diff(before, after, label='filas'):
    removed = before - after
    pct = removed / before * 100 if before > 0 else 0
    print(f'  Antes: {before:>8,} | Después: {after:>8,} | Eliminados: {removed:>6,} ({pct:.2f}%)')

print('✅ Setup completo')

✅ Setup completo


## 1. Carga de datos raw

In [2]:
customers   = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders      = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products    = pd.read_csv('../data/raw/olist_products_dataset.csv')
payments    = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews     = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
sellers     = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

print('✅ Datos raw cargados')

✅ Datos raw cargados


---
## 2. Utilidades compartidas

In [3]:
def normalize_text(s):
    """
    Normaliza strings:
      - Pasa a minúsculas
      - Elimina acentos (unicodedata NFKD)
      - Reemplaza espacios múltiples por uno solo
      - Strip de espacios al inicio/fin
    Justificación: los campos 'city' y 'state' tienen inconsistencias
    de encoding detectadas en exploración (ej: 'sao paulo' vs 'são paulo').
    """
    if pd.isna(s):
        return s
    s = str(s).lower().strip()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r'\s+', ' ', s)
    return s

print('✅ Función normalize_text definida')

✅ Función normalize_text definida


---
## 3. Limpieza por tabla

### 3.1 `geolocation`

**Hallazgos:**
- 1,000,163 filas con 981,148 duplicados por ZIP (98.10%) — múltiples coords por código postal
- 31 latitudes y 26 longitudes fuera del territorio de Brasil

**Decisiones:**
1. Filtrar coordenadas fuera del bounding box de Brasil (lat: -33.7 a 5.3 / lng: -73.9 a -28.8)
2. Agregar por ZIP usando la mediana de lat/lng — más robusta que la media ante outliers
3. Normalizar encoding de `geolocation_city`

In [4]:
section('geolocation')
before = len(geolocation)

# 1. Filtrar coordenadas fuera de Brasil
geo_clean = geolocation[
    (geolocation['geolocation_lat'].between(-33.7, 5.3)) &
    (geolocation['geolocation_lng'].between(-73.9, -28.8))
].copy()
log(f'Coordenadas fuera de Brasil eliminadas: {before - len(geo_clean)}')

# 2. Normalizar ciudad
geo_clean['geolocation_city'] = geo_clean['geolocation_city'].apply(normalize_text)

# 3. Agregar: mediana de lat/lng por ZIP + state/city más frecuente
geo_agg = (
    geo_clean
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat   = ('geolocation_lat',  'median'),
        geolocation_lng   = ('geolocation_lng',  'median'),
        geolocation_city  = ('geolocation_city',  lambda x: x.mode().iloc[0]),
        geolocation_state = ('geolocation_state', lambda x: x.mode().iloc[0]),
    )
)

log(f'Tabla agregada: 1 fila por ZIP code')
diff(before, len(geo_agg))
print(f'  ZIPs únicos resultantes: {len(geo_agg):,}')
geo_agg.head(3)


  geolocation
  ✔  Coordenadas fuera de Brasil eliminadas: 31
  ✔  Tabla agregada: 1 fila por ZIP code
  Antes: 1,000,163 | Después:   19,011 | Eliminados: 981,152 (98.10%)
  ZIPs únicos resultantes: 19,011


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550381,-46.634027,sao paulo,SP
1,1002,-23.548551,-46.635072,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP


---
### 3.2 `customers`

**Hallazgos:**
- customer_id es único (PK válida)
- customer_unique_id tiene 96,096 únicos vs 99,441 customer_id → compradores recurrentes
- Posibles problemas de encoding en `customer_city`

**Decisiones:**
1. Normalizar encoding en city
2. Estado a mayúsculas (formato estándar de 2 letras)

In [5]:
section('customers')
before = len(customers)

cust_clean = customers.copy()

# 1. Normalizar ciudad
cust_clean['customer_city'] = cust_clean['customer_city'].apply(normalize_text)
log('customer_city normalizado (encoding + lowercase)')

# 2. Estado a mayúsculas
cust_clean['customer_state'] = cust_clean['customer_state'].str.upper().str.strip()
log('customer_state estandarizado a mayúsculas')

# 3. Verificar que no hay nulos en columnas clave
assert cust_clean['customer_id'].isnull().sum() == 0, 'customer_id tiene nulos'
assert cust_clean.duplicated('customer_id').sum() == 0, 'customer_id tiene duplicados'
log('PKs verificadas: sin nulos ni duplicados')

diff(before, len(cust_clean))
cust_clean.head(3)


  customers
  ✔  customer_city normalizado (encoding + lowercase)
  ✔  customer_state estandarizado a mayúsculas
  ✔  PKs verificadas: sin nulos ni duplicados
  Antes:   99,441 | Después:   99,441 | Eliminados:      0 (0.00%)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


---
### 3.3 `sellers`

**Hallazgos:**
- Sin nulos, PK única
- Mismo problema de encoding en `seller_city`

**Decisiones:**
1. Normalizar encoding en city y state

In [6]:
section('sellers')
before = len(sellers)

sell_clean = sellers.copy()
sell_clean['seller_city']  = sell_clean['seller_city'].apply(normalize_text)
sell_clean['seller_state'] = sell_clean['seller_state'].str.upper().str.strip()
log('seller_city normalizado, seller_state en mayúsculas')

assert sell_clean.duplicated('seller_id').sum() == 0
log('PK verificada')

diff(before, len(sell_clean))
sell_clean.head(3)


  sellers
  ✔  seller_city normalizado, seller_state en mayúsculas
  ✔  PK verificada
  Antes:    3,095 | Después:    3,095 | Eliminados:      0 (0.00%)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


---
### 3.4 `products`

**Hallazgos:**
- 610 productos sin categoría (1.85%)
- Dimensiones físicas con posibles ceros/nulos
- Nombres de categoría en portugués

**Decisiones:**
1. Imputar categorías faltantes como `'unknown'` — no hay base para inferir categoría
2. Traducir categorías al inglés usando `category_translation`
3. Imputar dimensiones faltantes/cero con la mediana de su categoría — más representativo que la mediana global
4. Los ceros en dimensiones son imposibles físicamente → tratar igual que nulos

In [7]:
section('products')
before = len(products)

prod_clean = products.copy()

# 1. Imputar categorías faltantes
prod_clean['product_category_name'] = prod_clean['product_category_name'].fillna('unknown')
log(f'Categorías nulas imputadas como "unknown": {(products["product_category_name"].isna()).sum()}')

# 2. Traducir categorías al inglés
translation_map = dict(zip(
    category_translation['product_category_name'],
    category_translation['product_category_name_english']
))
translation_map['unknown'] = 'unknown'
prod_clean['product_category_name_english'] = (
    prod_clean['product_category_name']
    .map(translation_map)
    .fillna(prod_clean['product_category_name'])  # si no tiene traducción, mantener original
)
log(f'Categorías traducidas al inglés: {prod_clean["product_category_name_english"].nunique()} categorías únicas')

# 3. Tratar ceros como nulos en dimensiones físicas
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dim_cols:
    zeros = (prod_clean[col] == 0).sum()
    if zeros > 0:
        prod_clean[col] = prod_clean[col].replace(0, np.nan)
        log(f'{col}: {zeros} ceros convertidos a NaN')

# 4. Imputar dimensiones con mediana por categoría
for col in dim_cols:
    nulls_before = prod_clean[col].isna().sum()
    if nulls_before > 0:
        # Mediana por categoría
        cat_median = prod_clean.groupby('product_category_name')[col].transform('median')
        # Fallback: mediana global si la categoría también tiene todos nulos
        global_median = prod_clean[col].median()
        prod_clean[col] = prod_clean[col].fillna(cat_median).fillna(global_median)
        log(f'{col}: {nulls_before} nulos imputados con mediana por categoría')

# 5. Eliminar columna original en portugués (ya tenemos la traducida)
# Mantener ambas por trazabilidad
log('Ambas columnas de categoría conservadas (PT + EN)')

diff(before, len(prod_clean))
prod_clean[['product_id','product_category_name','product_category_name_english'] + dim_cols].head(3)


  products
  ✔  Categorías nulas imputadas como "unknown": 610
  ✔  Categorías traducidas al inglés: 74 categorías únicas
  ✔  product_weight_g: 4 ceros convertidos a NaN
  ✔  product_weight_g: 6 nulos imputados con mediana por categoría
  ✔  product_length_cm: 2 nulos imputados con mediana por categoría
  ✔  product_height_cm: 2 nulos imputados con mediana por categoría
  ✔  product_width_cm: 2 nulos imputados con mediana por categoría
  ✔  Ambas columnas de categoría conservadas (PT + EN)
  Antes:   32,951 | Después:   32,951 | Eliminados:      0 (0.00%)


,product_id,product_category_name,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,154.0,18.0,9.0,15.0


---
### 3.5 `orders`

**Hallazgos:**
- `order_approved_at`: 160 nulos (0.16%) — pedidos cancelados/pendientes
- `order_delivered_carrier_date`: 1,783 nulos (1.79%)
- `order_delivered_customer_date`: 2,965 nulos (2.98%)
- 775 órdenes sin ítems asociados
- 1 orden sin pago

**Decisiones:**
1. Parsear todas las fechas a `datetime`
2. Los nulos en fechas de entrega son **válidos** — pedidos cancelados, pendientes, etc. Se mantienen
3. Detectar y registrar anomalías temporales sin eliminarlas (pueden ser errores de sistema menores)
4. Crear columna `delivery_time_days` solo donde haya fecha completa
5. Crear columna `is_delivered` como flag binario
6. Las 775 órdenes sin ítems las mantenemos en la tabla orders pero las aislamos con un flag

In [8]:
section('orders')
before = len(orders)

ord_clean = orders.copy()

# 1. Parsear fechas
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in date_cols:
    ord_clean[col] = pd.to_datetime(ord_clean[col], errors='coerce')
log('Todas las columnas de fecha parseadas a datetime')

# 2. Flag: pedido entregado
ord_clean['is_delivered'] = (ord_clean['order_status'] == 'delivered').astype(int)
log(f'Flag is_delivered creado: {ord_clean["is_delivered"].sum():,} pedidos entregados')

# 3. Columna derivada: días hasta entrega (solo cuando hay fecha completa)
ord_clean['delivery_time_days'] = (
    ord_clean['order_delivered_customer_date'] - ord_clean['order_purchase_timestamp']
).dt.days
# Sanity check: eliminar valores negativos (imposibles físicamente)
negative_delivery = (ord_clean['delivery_time_days'] < 0).sum()
if negative_delivery > 0:
    ord_clean.loc[ord_clean['delivery_time_days'] < 0, 'delivery_time_days'] = np.nan
    log(f'delivery_time_days: {negative_delivery} valores negativos anulados')
log(f'delivery_time_days: media={ord_clean["delivery_time_days"].mean():.1f}d, '
    f'mediana={ord_clean["delivery_time_days"].median():.1f}d')

# 4. Columna derivada: días de retraso respecto a estimado
ord_clean['delay_days'] = (
    ord_clean['order_delivered_customer_date'] - ord_clean['order_estimated_delivery_date']
).dt.days
# Positivo = llegó tarde, Negativo = llegó antes
log(f'delay_days: {(ord_clean["delay_days"] > 0).sum():,} pedidos llegaron tarde')

# 5. Columnas temporales de análisis
ord_clean['order_year']  = ord_clean['order_purchase_timestamp'].dt.year
ord_clean['order_month'] = ord_clean['order_purchase_timestamp'].dt.month
ord_clean['order_yearmonth'] = ord_clean['order_purchase_timestamp'].dt.to_period('M').astype(str)
log('Columnas year, month, yearmonth creadas')

# 6. Flag: órdenes sin ítems (775 detectadas en exploración)
order_ids_with_items = set(order_items['order_id'])
ord_clean['has_items'] = ord_clean['order_id'].isin(order_ids_with_items).astype(int)
no_items = (ord_clean['has_items'] == 0).sum()
log(f'Flag has_items: {no_items} órdenes sin ítems marcadas (no eliminadas)')

diff(before, len(ord_clean))
ord_clean[['order_id','order_status','is_delivered','delivery_time_days','delay_days','order_yearmonth']].head(3)


  orders
  ✔  Todas las columnas de fecha parseadas a datetime
  ✔  Flag is_delivered creado: 96,478 pedidos entregados
  ✔  delivery_time_days: media=12.1d, mediana=10.0d
  ✔  delay_days: 6,535 pedidos llegaron tarde
  ✔  Columnas year, month, yearmonth creadas
  ✔  Flag has_items: 775 órdenes sin ítems marcadas (no eliminadas)
  Antes:   99,441 | Después:   99,441 | Eliminados:      0 (0.00%)


,order_id,order_status,is_delivered,delivery_time_days,delay_days,order_yearmonth
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,1,8.0,-8.0,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,1,13.0,-6.0,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,1,9.0,-18.0,2018-08


---
### 3.6 `order_items`

**Hallazgos:**
- `price`: mín 0.85, sin ceros ni negativos ✅
- `freight_value`: 383 ceros — posibles envíos gratuitos, no necesariamente errores
- Sin nulos

**Decisiones:**
1. Los freight_value == 0 se mantienen — son envíos gratuitos (práctica real en e-commerce)
2. Parsear `shipping_limit_date` a datetime
3. Crear columna `total_item_value = price + freight_value`

In [9]:
section('order_items')
before = len(order_items)

items_clean = order_items.copy()

# 1. Parsear fecha
items_clean['shipping_limit_date'] = pd.to_datetime(items_clean['shipping_limit_date'], errors='coerce')
log('shipping_limit_date parseado a datetime')

# 2. Columna derivada: valor total del ítem
items_clean['total_item_value'] = items_clean['price'] + items_clean['freight_value']
log(f'total_item_value creado: total revenue de ítems = BRL {items_clean["total_item_value"].sum():,.2f}')

# 3. Documentar los freight == 0 (sin eliminar)
free_shipping = (items_clean['freight_value'] == 0).sum()
log(f'freight_value == 0: {free_shipping} registros mantenidos (envíos gratuitos)')

diff(before, len(items_clean))
items_clean.head(3)


  order_items
  ✔  shipping_limit_date parseado a datetime
  ✔  total_item_value creado: total revenue de ítems = BRL 15,843,553.24
  ✔  freight_value == 0: 383 registros mantenidos (envíos gratuitos)
  Antes:  112,650 | Después:  112,650 | Eliminados:      0 (0.00%)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_item_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87,216.87


---
### 3.7 `payments`

**Hallazgos:**
- 9 pagos con `payment_value == 0`
- 2,961 órdenes con más de 1 registro de pago (hasta 29) — pago en múltiples cuotas/métodos
- 1 orden sin pago

**Decisiones:**
1. Eliminar los 9 registros con `payment_value == 0` — no representan transacción real
2. Los múltiples registros por orden son válidos (cuotas + distintos métodos) — mantener granularidad
3. Crear tabla adicional `payments_summary` con el total por orden (útil para joins)

In [10]:
section('payments')
before = len(payments)

pay_clean = payments.copy()

# 1. Eliminar pagos con valor cero
zero_payments = (pay_clean['payment_value'] == 0).sum()
pay_clean = pay_clean[pay_clean['payment_value'] > 0].copy()
log(f'Eliminados {zero_payments} registros con payment_value == 0')

# 2. Estandarizar payment_type a minúsculas
pay_clean['payment_type'] = pay_clean['payment_type'].str.lower().str.strip()
log(f'payment_type normalizado: {pay_clean["payment_type"].unique()}')

# 3. Tabla resumen: total pagado y método principal por orden
pay_summary = (
    pay_clean
    .groupby('order_id', as_index=False)
    .agg(
        total_payment        = ('payment_value',        'sum'),
        payment_installments = ('payment_installments', 'max'),
        payment_methods_count= ('payment_type',         'nunique'),
        primary_payment_type = ('payment_value',        lambda x: pay_clean.loc[x.idxmax(), 'payment_type']),
    )
)
log(f'payments_summary creado: {len(pay_summary):,} órdenes únicas')

diff(before, len(pay_clean))
pay_clean.head(3)


  payments
  ✔  Eliminados 9 registros con payment_value == 0
  ✔  payment_type normalizado: ['credit_card' 'boleto' 'voucher' 'debit_card']
  ✔  payments_summary creado: 99,437 órdenes únicas
  Antes:  103,886 | Después:  103,877 | Eliminados:      9 (0.01%)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


---
### 3.8 `reviews`

**Hallazgos:**
- `review_comment_title`: 87,656 nulos — la mayoría de reseñas no tienen título
- `review_comment_message`: 58,247 nulos — muchas sin texto
- 551 duplicados por `order_id`

**Decisiones:**
1. Los nulos en comentarios son **comportamiento normal** — se imputan con string vacío para facilitar análisis de texto
2. Para duplicados de order_id: quedarse con la reseña más reciente (`review_answer_timestamp` mayor) — es la más actualizada
3. Parsear fechas
4. Crear flag `is_negative` (score ≤ 2) y `has_comment`

In [11]:
section('reviews')
before = len(reviews)

rev_clean = reviews.copy()

# 1. Parsear fechas
rev_clean['review_creation_date']   = pd.to_datetime(rev_clean['review_creation_date'],   errors='coerce')
rev_clean['review_answer_timestamp']= pd.to_datetime(rev_clean['review_answer_timestamp'], errors='coerce')
log('Fechas de reviews parseadas')

# 2. Resolver duplicados por order_id: mantener la reseña más reciente
rev_clean = (
    rev_clean
    .sort_values('review_answer_timestamp', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
)
log(f'Duplicados por order_id resueltos: se mantuvo la reseña más reciente por orden')

# 3. Imputar comentarios nulos con string vacío
rev_clean['review_comment_title']   = rev_clean['review_comment_title'].fillna('')
rev_clean['review_comment_message'] = rev_clean['review_comment_message'].fillna('')
log('Nulos en comentarios imputados con string vacío')

# 4. Flags útiles
rev_clean['is_negative']  = (rev_clean['review_score'] <= 2).astype(int)
rev_clean['has_comment']  = (rev_clean['review_comment_message'].str.len() > 0).astype(int)
pct_neg = rev_clean['is_negative'].mean() * 100
log(f'Flag is_negative: {pct_neg:.2f}% de reseñas son negativas (score ≤ 2)')
log(f'Flag has_comment: {rev_clean["has_comment"].mean()*100:.1f}% de reseñas tienen comentario')

diff(before, len(rev_clean))
rev_clean[['order_id','review_score','is_negative','has_comment']].head(3)


  reviews
  ✔  Fechas de reviews parseadas
  ✔  Duplicados por order_id resueltos: se mantuvo la reseña más reciente por orden
  ✔  Nulos en comentarios imputados con string vacío
  ✔  Flag is_negative: 14.69% de reseñas son negativas (score ≤ 2)
  ✔  Flag has_comment: 41.3% de reseñas tienen comentario
  Antes:   99,224 | Después:   98,673 | Eliminados:    551 (0.56%)


,order_id,review_score,is_negative,has_comment
80582,30a2f24dd6770c91faa6b3481319204b,5,0,0
49615,7e8072dc0f35ebb0c1b2a4743e0f179a,5,0,0
92802,7ce4e38f4eadd993bb5b2e60bb7f7bec,5,0,1


---
## 4. Dataset maestro (fact table para análisis)

Join principal que une todas las tablas para facilitar el dashboard y las queries analíticas.

In [12]:
section('Master dataset')

# Base: solo órdenes con ítems (excluir las 775 huérfanas para análisis de negocio)
master = ord_clean[ord_clean['has_items'] == 1].copy()
log(f'Base: {len(master):,} órdenes con ítems')

# + customers
master = master.merge(cust_clean[['customer_id','customer_unique_id','customer_city','customer_state']],
                      on='customer_id', how='left')
log('Joined: customers')

# + payment summary
master = master.merge(pay_summary, on='order_id', how='left')
log('Joined: payments_summary')

# + reviews
master = master.merge(
    rev_clean[['order_id','review_score','is_negative','has_comment']],
    on='order_id', how='left'
)
log('Joined: reviews')

# + order_items (agrupado por order: suma de precio, promedio de flete, conteo de ítems)
items_agg = (
    items_clean
    .groupby('order_id', as_index=False)
    .agg(
        item_count      = ('order_item_id',    'count'),
        items_revenue   = ('price',            'sum'),
        freight_total   = ('freight_value',    'sum'),
        seller_id       = ('seller_id',        lambda x: x.mode().iloc[0]),  # vendedor principal
        product_id      = ('product_id',       lambda x: x.mode().iloc[0]),  # producto principal
    )
)
master = master.merge(items_agg, on='order_id', how='left')
log('Joined: order_items (agregado)')

# + products (categoría del producto principal)
master = master.merge(
    prod_clean[['product_id','product_category_name_english']],
    on='product_id', how='left'
)
log('Joined: products')

# + sellers
master = master.merge(
    sell_clean[['seller_id','seller_city','seller_state']],
    on='seller_id', how='left'
)
log('Joined: sellers')

print(f'\n  Shape final del master dataset: {master.shape}')
print(f'  Columnas: {list(master.columns)}')
master.head(3)


  Master dataset
  ✔  Base: 98,666 órdenes con ítems
  ✔  Joined: customers
  ✔  Joined: payments_summary
  ✔  Joined: reviews
  ✔  Joined: order_items (agregado)
  ✔  Joined: products
  ✔  Joined: sellers

  Shape final del master dataset: (98666, 33)
  Columnas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_delivered', 'delivery_time_days', 'delay_days', 'order_year', 'order_month', 'order_yearmonth', 'has_items', 'customer_unique_id', 'customer_city', 'customer_state', 'total_payment', 'payment_installments', 'payment_methods_count', 'primary_payment_type', 'review_score', 'is_negative', 'has_comment', 'item_count', 'items_revenue', 'freight_total', 'seller_id', 'product_id', 'product_category_name_english', 'seller_city', 'seller_state']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time_days,...,is_negative,has_comment,item_count,items_revenue,freight_total,seller_id,product_id,product_category_name_english,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,8.0,...,0.0,1.0,1,29.99,8.72,3504c0cb71d7fa48d967e0e4c94d59d9,87285b34884572647811a353c7ac498a,housewares,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,13.0,...,0.0,1.0,1,118.70,22.76,289cdb325fb7e7f891c38608bf9e0962,595fac2a385ac33a80bd5114aec74eb8,perfumery,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,9.0,...,0.0,0.0,1,159.90,19.22,4869f7a5dfa277a7dca6462dcf3b52b2,aa4383b373c6aca5d8797843e5594415,auto,guariba,SP


---
## 5. Validaciones finales

In [13]:
section('Validaciones finales')

tables_clean = {
    'customers':       cust_clean,
    'sellers':         sell_clean,
    'products':        prod_clean,
    'orders':          ord_clean,
    'order_items':     items_clean,
    'payments':        pay_clean,
    'payments_summary':pay_summary,
    'reviews':         rev_clean,
    'geolocation':     geo_agg,
    'master':          master,
}

print(f'{'Tabla':<22} {'Filas':>10} {'Columnas':>10} {'Nulos totales':>15}')
print('-' * 62)
for name, df in tables_clean.items():
    nulls = df.isnull().sum().sum()
    print(f'{name:<22} {len(df):>10,} {df.shape[1]:>10} {nulls:>15,}')

# Verificaciones críticas
print('\n--- Verificaciones críticas ---')
assert cust_clean.duplicated('customer_id').sum() == 0,  'customers: PK duplicada'
assert sell_clean.duplicated('seller_id').sum()  == 0,   'sellers: PK duplicada'
assert prod_clean.duplicated('product_id').sum() == 0,   'products: PK duplicada'
assert ord_clean.duplicated('order_id').sum()    == 0,   'orders: PK duplicada'
assert rev_clean.duplicated('order_id').sum()    == 0,   'reviews: duplicados por order_id'
assert (pay_clean['payment_value'] <= 0).sum()   == 0,   'payments: valores <= 0'
assert geo_agg.duplicated('geolocation_zip_code_prefix').sum() == 0, 'geo: ZIP duplicado'
print('  ✅ Todas las verificaciones pasaron')


  Validaciones finales
Tabla                       Filas   Columnas   Nulos totales
--------------------------------------------------------------
customers                  99,441          5               0
sellers                     3,095          4               0
products                   32,951         10           1,830
orders                     99,441         15          10,838
order_items               112,650          8               0
payments                  103,877          5               0
payments_summary           99,437          5               0
reviews                    98,673          9               0
geolocation                19,011          5               0
master                     98,666         33           9,844

--- Verificaciones críticas ---
  ✅ Todas las verificaciones pasaron


---
## 6. Exportar CSVs limpios

In [14]:
section('Exportando datos limpios')

export_map = {
    '../data/clean/customers_clean.csv':        cust_clean,
    '../data/clean/sellers_clean.csv':          sell_clean,
    '../data/clean/products_clean.csv':         prod_clean,
    '../data/clean/orders_clean.csv':           ord_clean,
    '../data/clean/order_items_clean.csv':      items_clean,
    '../data/clean/payments_clean.csv':         pay_clean,
    '../data/clean/payments_summary_clean.csv': pay_summary,
    '../data/clean/reviews_clean.csv':          rev_clean,
    '../data/clean/geolocation_clean.csv':      geo_agg,
    '../data/clean/master_dataset.csv':         master,
}

for path, df in export_map.items():
    df.to_csv(path, index=False)
    print(f'  ✔  {path.split("/")[-1]:<40} {len(df):>10,} filas')

print('\n✅ Todos los archivos exportados a data/clean/')


  Exportando datos limpios
  ✔  customers_clean.csv                          99,441 filas
  ✔  sellers_clean.csv                             3,095 filas
  ✔  products_clean.csv                           32,951 filas
  ✔  orders_clean.csv                             99,441 filas
  ✔  order_items_clean.csv                       112,650 filas
  ✔  payments_clean.csv                          103,877 filas
  ✔  payments_summary_clean.csv                   99,437 filas
  ✔  reviews_clean.csv                            98,673 filas
  ✔  geolocation_clean.csv                        19,011 filas
  ✔  master_dataset.csv                           98,666 filas

✅ Todos los archivos exportados a data/clean/


---
## 7. Resumen de decisiones de limpieza

| Tabla | Problema | Decisión | Justificación |
|-------|----------|----------|-----------------|
| geolocation | 57 coords fuera de Brasil | Filtrar por bounding box | Imposibles geográficamente |
| geolocation | 981k duplicados por ZIP | Agregar con mediana lat/lng | 1 coord representativa por ZIP |
| customers / sellers | Encoding en city | normalize_text() | Consistencia para joins y filtros |
| products | 610 sin categoría | Imputar `'unknown'` | No hay base para inferir categoría |
| products | Dimensiones = 0 | Tratar como nulo + imputar mediana por categoría | Imposibles físicamente |
| products | Categorías en PT | Traducir al inglés | Mejor legibilidad en dashboard |
| orders | Nulos en fechas de entrega | Mantener | Son válidos (cancelados/pendientes) |
| orders | 775 sin ítems | Flag `has_items` | Preservar para auditoría |
| order_items | freight == 0 | Mantener | Envíos gratuitos son práctica real |
| payments | 9 payment_value == 0 | Eliminar | No representan transacción real |
| reviews | 551 duplicados order_id | Mantener más reciente | La reseña actualizada es la vigente |
| reviews | Nulos en comentarios | Imputar con `''` | Son válidos; facilita análisis de texto |

**Próximo paso →** `03_schema.sql` + carga a base de datos.